In [24]:
import re
import pretty_midi

# --- Parámetros globales ---
root_octave = 4
chord_duration = 1.5
velocity = 90

# --- Diccionarios base ---
NOTE_TO_PC = {
    'C': 0, 'C#': 1, 'DB': 1, 'D': 2, 'D#': 3, 'EB': 3, 'E': 4, 'F': 5,
    'F#': 6, 'GB': 6, 'G': 7, 'G#': 8, 'AB': 8, 'A': 9, 'A#': 10, 'BB': 10, 'B': 11
}

CHORD_TEMPLATES = {
    '':      [0, 4, 7],
    'maj':   [0, 4, 7],
    'major': [0, 4, 7],
    'min':   [0, 3, 7],
    'm':     [0, 3, 7],
    '7':     [0, 4, 7, 10],
    'maj7':  [0, 4, 7, 11],
    'm7':    [0, 3, 7, 10],
    'min7':  [0, 3, 7, 10],
    'dim':   [0, 3, 6],
    'dim7':  [0, 3, 6, 9],
    'm7b5':  [0, 3, 6, 10],
    'aug':   [0, 4, 8],
    '9':     [0, 4, 7, 10, 14],
    'maj9':  [0, 4, 7, 11, 14],
    'sus2':  [0, 2, 7],
    'sus4':  [0, 5, 7]
}

# --- Funciones auxiliares ---
def parse_chord_symbol(sym):
    """Convierte símbolo de acorde en pitch classes"""
    s = sym.strip().replace('–', '-').replace('—', '-')
    s = s.replace(' ', '').upper()
    m = re.match(r'^([A-G])([#B]?)(.*)$', s)
    if not m:
        return None
    root_letter = m.group(1)
    accidental = m.group(2)
    quality = m.group(3)
    root_name = root_letter + accidental
    if root_name not in NOTE_TO_PC:
        return None
    candidates = sorted(CHORD_TEMPLATES.keys(), key=lambda x: -len(x))
    chosen = None
    for c in candidates:
        if quality.startswith(c.upper()):
            chosen = c
            break
    if chosen is None:
        chosen = ''
    return {
        'root_pc': NOTE_TO_PC[root_name],
        'template': CHORD_TEMPLATES[chosen],
        'chosen_quality': chosen
    }

def chord_pitches_from_template(root_pc, template, octave):
    base = root_pc + 12 * octave
    pitches = []
    for interval in template:
        pitch = base + interval
        if 0 <= pitch <= 127:
            pitches.append(int(pitch))
    return pitches

def safe_filename(s):
    """Reemplaza caracteres problemáticos en nombres de archivo"""
    return s.replace("#", "s").replace("/", "_").replace("\\", "_")

def progression_to_midi_file(prefix, prog_id, chord_list):
    """Crea archivo MIDI de una progresión"""
    midi = pretty_midi.PrettyMIDI()
    piano = pretty_midi.Instrument(program=0)
    t = 0.0
    for sym in chord_list:
        parsed = parse_chord_symbol(sym)
        if parsed is None:
            print(f"⚠️ No pude parsear '{sym}' — lo salto.")
            t += chord_duration
            continue
        pitches = chord_pitches_from_template(parsed['root_pc'], parsed['template'], root_octave)
        for p in pitches:
            note = pretty_midi.Note(velocity=velocity, pitch=p, start=t, end=t + chord_duration)
            piano.notes.append(note)
        t += chord_duration

    midi.instruments.append(piano)
    chords_str = "-".join([safe_filename(c) for c in chord_list])
    filename = f"{prefix}{prog_id}) {chords_str}.mid"
    midi.write(filename)
    print(f"✅ Generado: {filename}")

# --- Progresiones tipo L ---
l_progressions = {
    1: ["Cmin", "Gmaj", "Cmin", "Fmin", "Cmin"],
    4: ["Cm", "C7", "Fm", "Bb", "C"],
    3: ["Cmin", "Gmaj", "A7", "Fmaj", "Cmin"],
    10: ["C#", "F#dim7", "Fm7", "Bb9", "F"],
    5: ["Cmin", "Emin", "Emin", "Fmaj", "Cmin"],
    9: ["Abmaj7", "Cmin7", "C#maj", "Ebmaj"]
}

# --- Progresiones tipo M ---
m_progressions = {
    3: ["Cm", "Bb", "Ab", "C#", "Eb"],
    2: ["Cmin", "Gmaj", "C7", "Fmin", "Cmin"],
    22: ["Cm", "Bb", "Bb", "Dm7", "Eb"],  # duplicado
    4: ["Cmin", "Gmaj", "F#7", "F7", "Cmin"],
    1: ["Cm", "Bb", "Bb", "Cm7", "Fm7"],
    6: ["Cmin", "F#min", "A7", "D#min", "Cmin"]
}

# --- Generar archivos MIDI ---
print("🎹 Generando progresiones tipo L...")
for pid, chords in l_progressions.items():
    progression_to_midi_file("L", pid, chords)

print("\n🎹 Generando progresiones tipo M...")
for pid, chords in m_progressions.items():
    progression_to_midi_file("M", pid, chords)




🎹 Generando progresiones tipo L...
✅ Generado: L1) Cmin-Gmaj-Cmin-Fmin-Cmin.mid
✅ Generado: L4) Cm-C7-Fm-Bb-C.mid
✅ Generado: L3) Cmin-Gmaj-A7-Fmaj-Cmin.mid
✅ Generado: L10) Cs-Fsdim7-Fm7-Bb9-F.mid
✅ Generado: L5) Cmin-Emin-Emin-Fmaj-Cmin.mid
✅ Generado: L9) Abmaj7-Cmin7-Csmaj-Ebmaj.mid

🎹 Generando progresiones tipo M...
✅ Generado: M3) Cm-Bb-Ab-Cs-Eb.mid
✅ Generado: M2) Cmin-Gmaj-C7-Fmin-Cmin.mid
✅ Generado: M22) Cm-Bb-Bb-Dm7-Eb.mid
✅ Generado: M4) Cmin-Gmaj-Fs7-F7-Cmin.mid
✅ Generado: M1) Cm-Bb-Bb-Cm7-Fm7.mid
✅ Generado: M6) Cmin-Fsmin-A7-Dsmin-Cmin.mid


In [5]:
# Importar CSV
import csv
import pandas as pd

file = open('2.csv', 'r')
reader = csv.reader(file)
data = list(reader)

df = pd.DataFrame(data[1:], columns=data[0])
print(df.head())

                                                      \
0  /Users/pepebeats/Desktop/Semi-Supervised-CVAE-...   
1  /Users/pepebeats/Desktop/Semi-Supervised-CVAE-...   
2  /Users/pepebeats/Desktop/Semi-Supervised-CVAE-...   
3  /Users/pepebeats/Desktop/Semi-Supervised-CVAE-...   
4  /Users/pepebeats/Desktop/Semi-Supervised-CVAE-...   

  Vertical_Interval_Histogram_0 Vertical_Interval_Histogram_1  \
0                           0.0                           0.0   
1                           0.0                           0.0   
2                           0.0                           0.0   
3                           0.0                           0.0   
4                           0.0                           0.0   

  Vertical_Interval_Histogram_2 Vertical_Interval_Histogram_3  \
0                           0.0                        0.3333   
1                           0.0                        0.2778   
2                           0.0                        0.3333   
3           

In [6]:
# Dividir datos por los nombres de archivo, formato L1), L2), etc. y el otro grupo M1
df_l = df[df[''].str.contains('L1|L3|L4|L5|L9|L10')]    
df_m = df[df[''].str.contains('M1|M2|M3|M4|M6|M22')]

# tomar L1 y no L10
df_base = df[df[''].str.contains('L1') & ~df[''].str.contains('L10')]


print("Datos L:")
print(df_l)

print("Datos M:")
print(df_m)

print("Datos Base:")
print(df_base)

Datos L:
                                                       \
0   /Users/pepebeats/Desktop/Semi-Supervised-CVAE-...   
1   /Users/pepebeats/Desktop/Semi-Supervised-CVAE-...   
3   /Users/pepebeats/Desktop/Semi-Supervised-CVAE-...   
6   /Users/pepebeats/Desktop/Semi-Supervised-CVAE-...   
8   /Users/pepebeats/Desktop/Semi-Supervised-CVAE-...   
11  /Users/pepebeats/Desktop/Semi-Supervised-CVAE-...   

   Vertical_Interval_Histogram_0 Vertical_Interval_Histogram_1  \
0                            0.0                           0.0   
1                            0.0                           0.0   
3                            0.0                           0.0   
6                            0.0                           0.0   
8                            0.0                           0.0   
11                           0.0                           0.0   

   Vertical_Interval_Histogram_2 Vertical_Interval_Histogram_3  \
0                            0.0                        0.3333

In [ ]:
# Pasar las columnas a tipo float y calcular media y desviacion estandar, menos la columna de nombres
mean_l = df_l.iloc[:, 1:].astype(float).mean()
std_l = df_l.iloc[:, 1:].astype(float).std()
mean_m = df_m.iloc[:, 1:].astype(float).mean()
std_m = df_m.iloc[:, 1:].astype(float).std()
mean_base = df_base.iloc[:, 1:].astype(float).mean()
std_base = df_base.iloc[:, 1:].astype(float).std()

# Eliminar todas las columnas que sub contengan "Histogram" 
histogram_columns = [col for col in df_l.columns if "Histogram" in col]
df_l = df_l.drop(columns=histogram_columns)
df_m = df_m.drop(columns=histogram_columns)
df_base = df_base.drop(columns=histogram_columns)   


# Imprimir por pantalla como tabla con +- desviacion estandar
print("Datos Base:")
for col in mean_base.index:
    print(f"{col}: {mean_base[col]:.4f} ± {std_base[col]:.4f}")

print("Datos literatura:")
for col in mean_l.index:
    print(f"{col}: {mean_l[col]:.4f} ± {std_l[col]:.4f}")
print("\nDatos mio:")
for col in mean_m.index:
    print(f"{col}: {mean_m[col]:.4f} ± {std_m[col]:.4f}")


Datos Base:
Average_Number_of_Simultaneous_Pitch_Classes: 3.0000 ± nan
Variability_of_Number_of_Simultaneous_Pitch_Classes: 0.0000 ± nan
Average_Number_of_Simultaneous_Pitches: 3.0000 ± nan
Variability_of_Number_of_Simultaneous_Pitches: 0.0000 ± nan
Most_Common_Vertical_Interval: 3.0000 ± nan
Second_Most_Common_Vertical_Interval: 4.0000 ± nan
Distance_Between_Two_Most_Common_Vertical_Intervals: 1.0000 ± nan
Prevalence_of_Most_Common_Vertical_Interval: 0.3333 ± nan
Prevalence_of_Second_Most_Common_Vertical_Interval: 0.3333 ± nan
Prevalence_Ratio_of_Two_Most_Common_Vertical_Intervals: 1.0000 ± nan
Vertical_Unisons: 0.0000 ± nan
Vertical_Minor_Seconds: 0.0000 ± nan
Vertical_Thirds: 0.6667 ± nan
Vertical_Tritones: 0.0000 ± nan
Vertical_Perfect_Fourths: 0.0000 ± nan
Vertical_Perfect_Fifths: 0.3333 ± nan
Vertical_Sixths: 0.0000 ± nan
Vertical_Sevenths: 0.0000 ± nan
Vertical_Octaves: 0.0000 ± nan
Perfect_Vertical_Intervals: 0.3333 ± nan
Vertical_Dissonance_Ratio: 0.0000 ± nan
Vertical_Minor_T